# E-Test: Untersuchung der Immanenz

Untersucht die Verteilung von Primzahlen mod 12 und deren Rolle in arithmetischen Progressionen (Konfigurationen).

In [1]:
from sage.all import *

In [2]:
def nisp_four_squares(n):
    """
    Hilfsfunktion für die Darstellung als Summe von 4 Quadraten.
    """
    return late_four_squares(n)

def late_four_squares(n):
    """
    Findet eine Darstellung von n als Summe von 4 Quadraten.
    Nutzt Sage's interne Funktionen.
    """
    try:
        # Konvertiere zu Sage Integer falls nötig
        if not isinstance(n, Integer):
            n = Integer(n)
        
        # Versuche SageMath-Methode
        if hasattr(n, 'is_sum_of_four_squares'):
            result = n.is_sum_of_four_squares(ext=True)
            if result:
                return list(result)
        
        # Fallback: Klassischer Algorithmus
        n_val = int(n)
        if n_val == 0:
            return [0, 0, 0, 0]
        
        sqrt_n = int(sqrt(n_val))
        for e in range(sqrt_n, -1, -1):
            remainder = n_val - e**2
            if remainder == 0:
                return [e, 0, 0, 0]
            
            sqrt_r = int(sqrt(remainder))
            for a in range(sqrt_r, -1, -1):
                remainder2 = remainder - a**2
                if remainder2 == 0:
                    return [e, a, 0, 0]
                
                sqrt_r2 = int(sqrt(remainder2))
                for b in range(sqrt_r2, -1, -1):
                    remainder3 = remainder2 - b**2
                    if remainder3 >= 0:
                        c = int(sqrt(remainder3))
                        if c**2 == remainder3:
                            return [e, a, b, c]
        
        return None
        
    except Exception as e:
        print(f"Fehler bei der Vier-Quadrate-Zerlegung: {e}")
        return None

In [3]:
def untersuche_immanenz(limit=1000):
    """
    Untersucht die Verteilung von Primzahlen mod 12 und deren Rolle in 
    arithmetischen Progressionen (Konfigurationen).
    """
    primes_12 = {1: [], 5: [], 7: [], 11: []}
    
    # 1. Primzahlen klassifizieren
    for p in primes(limit):
        if p > 3:
            primes_12[p % 12].append(p)
    
    print(f"Analyse der Primzahlen bis {limit}:")
    for r in [1, 5, 7, 11]:
        print(f"  Klasse {r} mod 12: {len(primes_12[r])} Primzahlen")

    # 2. Suche nach arithmetischen Progressionen der Länge 3 (AP-3)
    # Eine "Konfiguration" im Sinne des Dokuments
    ap3_konfigurationen = []
    p_set = set(primes(limit * 2))
    
    for r in [1, 5, 7, 11]:
        for i in range(len(primes_12[r])):
            p1 = primes_12[r][i]
            for j in range(i + 1, len(primes_12[r])):
                p2 = primes_12[r][j]
                diff = p2 - p1
                p3 = p2 + diff
                if p3 in p_set:
                    ap3_konfigurationen.append((p1, p2, p3))

    # 3. Prüfung der "Variationsfreiheit" (Lemma 1 & 3)
    # Wir schauen, ob es Konfigurationen gibt, die NUR aus der Elferklasse bestehen
    elfer_aps = [ap for ap in ap3_konfigurationen if all(x % 12 == 11 for x in ap)]
    
    print(f"\nErgebnis der Konfigurationssuche:")
    print(f"  Gefundene AP-3 insgesamt: {len(ap3_konfigurationen)}")
    print(f"  AP-3 ausschließlich in der 'Elferklasse' (11 mod 12): {len(elfer_aps)}")
    
    if len(elfer_aps) > 0:
        print(f"  Beispiel einer 'reinen' Elfer-Konfiguration: {elfer_aps[0]}")
        print("  Kritik: Da reine Elfer-Konfigurationen existieren, ist die Behauptung der")
        print("  notwendigen Einbindung von 1, 5, 7 (Lemma 1) rein numerisch schwer haltbar.")

    # 4. Brücke zur #Energiedoku: Quaternionen-Repräsentation
    # Jede Primzahl p kann als Norm eines Quaternions p = a^2 + b^2 + c^2 + d^2 geschrieben werden.
    print("\nStruktureller Check via Quaternionen (Norm-Darstellung):")
    for p_test in primes_12[11][:3]: # Teste die ersten drei der Elferklasse
        # Vier-Quadrate-Satz (Darstellung finden)
        sol = nisp_four_squares(p_test)
        print(f"  Primzahl {p_test} (11 mod 12) als Quaternion-Norm: {sol}")

In [4]:
# Ausführung
untersuche_immanenz(500)

Analyse der Primzahlen bis 500:
  Klasse 1 mod 12: 21 Primzahlen
  Klasse 5 mod 12: 23 Primzahlen
  Klasse 7 mod 12: 24 Primzahlen
  Klasse 11 mod 12: 25 Primzahlen

Ergebnis der Konfigurationssuche:
  Gefundene AP-3 insgesamt: 444
  AP-3 ausschließlich in der 'Elferklasse' (11 mod 12): 126
  Beispiel einer 'reinen' Elfer-Konfiguration: (11, 47, 83)
  Kritik: Da reine Elfer-Konfigurationen existieren, ist die Behauptung der
  notwendigen Einbindung von 1, 5, 7 (Lemma 1) rein numerisch schwer haltbar.

Struktureller Check via Quaternionen (Norm-Darstellung):
  Primzahl 11 (11 mod 12) als Quaternion-Norm: [3, 1, 1, 0]
  Primzahl 23 (11 mod 12) als Quaternion-Norm: [3, 3, 2, 1]
  Primzahl 47 (11 mod 12) als Quaternion-Norm: [6, 3, 1, 1]
